[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/08_data_vis/08_2_Choosing_a_Chart.ipynb)

# 08.2: Choosing a Chart

Notebook 08.1 showed how to draw some charts, and this notebook is about which one to draw. Python will draw almost anything you ask for, whether or not it makes sense, so the choice is up to you. We add two chart types for looking at relationships and change over time, and then practice matching a chart to a question and describing what it shows.

## The data

We use the crime data from 08.1.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent", "Rates.Property.All": "Property"})
crime = crime[["State", "Year", "Population", "Violent", "Property"]]
places_2012 = crime[(crime["Year"] == 2012) & (crime["State"] != "United States")]
places_2012.head(3)

## Scatter plots: are two numeric columns related?

A **scatter plot** draws one point per row, with one numeric column on each axis. Do states with higher violent crime rates also have higher property crime rates?

In [ ]:
ax = sns.scatterplot(data=places_2012, x="Violent", y="Property")
ax.set_title("Violent and property crime rates by state, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("Property crimes per 100,000 people")
plt.show()

Each point is one state. The points drift upward from left to right, so states with higher violent crime rates tend to have higher property crime rates. The relationship is far from exact, and one point sits far to the upper right, which is the District of Columbia. A number called the **correlation** summarizes how closely two columns move together. It runs from -1 to 1, where values near 1 mean they rise together, values near -1 mean one falls as the other rises, and values near 0 mean there is no straight-line relationship. `.corr()` calculates it.

In [ ]:
print(places_2012["Violent"].corr(places_2012["Property"]))
without_dc = places_2012[places_2012["State"] != "District of Columbia"]
print(without_dc["Violent"].corr(without_dc["Property"]))

With every state and D.C. the correlation is about 0.67, and without D.C. it drops to about 0.53. One unusual point can noticeably change a correlation, which is a good reason to look at the scatter plot and not only the number. A correlation also says nothing about the cause: neither kind of crime is shown here to produce the other.

## Line plots: how does something change over time?

A **line plot** connects points in order, which suits data with a natural sequence such as years. Here is the violent crime rate in Illinois.

In [ ]:
illinois = crime[crime["State"] == "Illinois"]
ax = sns.lineplot(data=illinois, x="Year", y="Violent")
ax.set_title("Violent crime rate in Illinois, 1960-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

The rate climbs from 365.1 in 1960 to a peak of 1039.2 in 1991, and then falls to 414.8 by 2012. To compare two lines, make two calls and give each one a `label=`, and then add a legend. Both calls draw on the same chart.

In [ ]:
nation = crime[crime["State"] == "United States"]
ax = sns.lineplot(data=illinois, x="Year", y="Violent", label="Illinois")
sns.lineplot(data=nation, x="Year", y="Violent", label="United States")
ax.set_title("Violent crime rates, 1960-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
ax.legend()
plt.show()

Both lines rise and then fall, and the Illinois line stays above the national line in all 53 years. The gap was 281.0 at the 1991 peak and only 27.9 by 2012. You could pull those numbers out of a table, but the chart shows the whole history at once.

## Coloring by a category: `hue`

`hue=` colors the points or bars by a category. This is another way of adding a second category to a chart. The next chart asks whether large states differ from small ones, using a category that we make with a mask.

In [ ]:
places = places_2012.copy()
places["Large"] = places["Population"] > 5000000
ax = sns.scatterplot(data=places, x="Violent", y="Property", hue="Large")
ax.set_title("Crime rates and state size, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("Property crimes per 100,000 people")
plt.show()

The two colors are mixed together across the chart, so the size of a state does not seem to separate high-crime states from low-crime ones in this data.

## When a chart does not fit

Here is a chart with a real problem. A bar for each of the 51 places, in alphabetical order, cannot be read, and the labels pile on top of each other.

In [ ]:
ax = sns.barplot(data=places_2012, x="State", y="Violent")
ax.set_title("Too many bars, in no useful order")
plt.show()

The fix has three parts: keep only the rows that matter to your question, sort them so that the order means something, and switch the axes so that the names are readable. To make a horizontal bar chart, put the category on `y`.

In [ ]:
top_ten = places_2012.sort_values("Violent", ascending=False).head(10)
ax = sns.barplot(data=top_ten, x="Violent", y="State")
ax.set_title("Ten highest violent crime rates, 2012")
ax.set_xlabel("Violent crimes per 100,000 people")
ax.set_ylabel("")
plt.show()

Now you can read it immediately: the District of Columbia is about twice the rate of Tennessee, which is next. Values like the ones in Broadway `Gross` create a different problem, since an axis may show numbers such as `1e9`. The simplest fix is to change the data, by plotting a column in millions or billions.

## Which chart for which question?

| Question | Chart | Function |
|---|---|---|
| What is the shape of one numeric column? | Histogram or box plot | `sns.histplot`, `sns.boxplot` |
| How do categories compare on a number? | Bar chart | `sns.barplot` |
| How many rows are in each category? | Count plot | `sns.countplot` |
| Are two numeric columns related? | Scatter plot | `sns.scatterplot` |
| How does something change over time? | Line plot | `sns.lineplot` |

A chart is finished when someone else could read it, which means it has a title and labeled axes, including units. Saying what it shows is part of the job too. A useful description names the pattern and gives numbers, as in "the Illinois rate peaked at 1039.2 in 1991 and fell to 414.8 by 2012."

## Reusing a chart with a function

If you are making the same chart for several states, a function saves you from repeating yourself, as functions did in chapter 4. This one takes the name of a state, and it does not need anything new.

In [ ]:
def plot_state(state_name):
    rows = crime[crime["State"] == state_name]
    ax = sns.lineplot(data=rows, x="Year", y="Violent")
    ax.set_title("Violent crime rate in " + state_name)
    ax.set_xlabel("Year")
    ax.set_ylabel("Violent crimes per 100,000 people")
    plt.show()

plot_state("Vermont")

## What's next

That finishes the notebooks for the course. You can now load a dataset, check it, select from it, summarize it, compare groups and show your results in a chart. These are the core skills of working with data, and CS/DS 225 builds on them with more detailed work on cleaning, grouping and visualization.